In [1]:
import requests
import os
import time
import glob
import random

OUT_DIR = "data/raw"
os.makedirs(OUT_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "OrchardLM/1.0 (https://github.com/ArthGV/orchardlm; arthemeg@gmail.com)"
}

In [29]:
# ── Fetch a single article ─────────────────────────────────────────────────────

def fetch_article(title: str) -> str | None:
    url    = "https://en.wikipedia.org/w/api.php"
    params = {
        "action":      "query",
        "format":      "json",
        "titles":      title,
        "prop":        "extracts",
        "explaintext": True,
        "exsectionformat": "plain",
    }
    response = requests.get(url, params=params, headers=HEADERS)
    pages    = response.json()["query"]["pages"]
    page     = next(iter(pages.values()))
    if "extract" not in page:
        return None
    return page["extract"]

# ── Get a list of articles to fetch ───────────────────────────────────────────

def get_article_titles(category: str, limit: int = 100) -> list[str]:
    url    = "https://en.wikipedia.org/w/api.php"
    titles = []
    params = {
        "action":  "query",
        "format":  "json",
        "list":    "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmlimit": min(limit, 500),
        "cmtype":  "page",
    }

    while len(titles) < limit:
        response = requests.get(url, params=params, headers=HEADERS)
        if response.status_code != 200 or not response.text:
            print(f"Bad response: {response.status_code}")
            break
        data = response.json()
        if "query" not in data:
            print(f"Unexpected: {data}")
            break
        titles += [m["title"] for m in data["query"]["categorymembers"]]
        if "continue" not in data or len(titles) >= limit:
            break
        params["cmcontinue"] = data["continue"]["cmcontinue"]

    return titles[:limit]

# ── Clean the text ─────────────────────────────────────────────────────────────

def clean_article(text: str) -> str:
    """Remove section headers, references, and empty lines."""
    lines   = text.split("\n")
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line:                          # skip empty lines
            continue
        if line.startswith("=="):             # skip section headers
            continue
        if line.startswith("^") or line.startswith("↑"):  # skip references
            continue
        if len(line) < 40:                    # skip very short lines (captions etc)
            continue
        cleaned.append(line)
    return "\n".join(cleaned)

# ── Main scraper ───────────────────────────────────────────────────────────────

def scrape_wikipedia(categories: list[str], articles_per_category: int = 50) -> None:
    all_titles_by_category = {}
    for category in categories:
        titles = get_article_titles(category, limit=articles_per_category)
        print(f"Category '{category}': {len(titles)} articles")
        all_titles_by_category[category] = titles

    for category, titles in all_titles_by_category.items():
        category_dir = f"{OUT_DIR}/{category.replace(' ', '_').lower()}"
        os.makedirs(category_dir, exist_ok=True)

        for title in titles:
            out_path = f"{category_dir}/{title.replace('/', '_')}.txt"
            if os.path.exists(out_path):
                print(f"  skip: {title}")
                continue

            text = fetch_article(title)
            if text:
                text = clean_article(text)
                with open(out_path, "w", encoding="utf-8") as f:
                    f.write(text)
                print(f"  saved: {title} ({len(text):,} chars)")
            else:
                print(f"  failed: {title}")

            time.sleep(0.5) #being polite

# ── Build final corpus ─────────────────────────────────────────────────────────

def build_corpus(raw_dir: str, out_path: str, max_chars: int = 10_000_000) -> None:
    total = 0
    with open(out_path, "w", encoding="utf-8") as out:
        files = glob.glob(raw_dir + '/*.txt')
        print(files)
        random.shuffle(files)
        for fpath in files:
            if not fpath.endswith(".txt"):
                continue
            with open(f"{fpath}", "r", encoding="utf-8") as f:
                text = f.read()
            out.write(text + "\n\n")
            total += len(text)
            if total >= max_chars:
                print(f"Reached limit at {fpath}")
                break

    print(f"Corpus: {total:,} chars → {out_path}")

In [30]:
#── Run ────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    categories = [
        "apple"
    ]

    scrape_wikipedia(categories, articles_per_category=10)
    build_corpus("data/raw/fruit", "data/corpus.txt", max_chars=10_00_000_000)

Category 'apple': 0 articles
['data/raw/fruit/Monkey orange.txt', 'data/raw/fruit/Fruit stand.txt', 'data/raw/fruit/Apeel Sciences.txt', 'data/raw/fruit/Bletting.txt', 'data/raw/fruit/Pomology.txt', 'data/raw/fruit/Climacteric (botany).txt', 'data/raw/fruit/Fruit.txt', 'data/raw/fruit/Arrouya.txt', 'data/raw/fruit/Tropical fruit.txt']
Corpus: 43,428 chars → data/corpus.txt
